# Nigeria Forest Monitor

Unified workflow for Sentinel-1 exploration, change detection, feature-model training, grid risk scoring, and alert outputs.

> Risk scores are decision-support indicators, not proof of hostile activity. Review and corroborate every alert.

## 1. Setup

Load the project environment and shared modules. Select the project `.venv` kernel before running all cells.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import ee
import pandas as pd

from src.config import load_config
from src.ingestion.gee_download import (
    init_gee, build_aoi, get_s1_collection,
    build_monthly_composites, median_composite, preview_composite,
)
from src.ingestion.grid import create_grid, tag_zones, preview_grid
from src.ingestion.acled_fetch import (
    load_acled, filter_to_aoi, incident_summary,
    tag_incidents_to_grid, compute_proximity_scores,
)
from src.preprocessing.baseline import build_baseline, preview_baseline
from src.preprocessing.speckle_filter import preview_filter_comparison, filter_composite
from src.detection.change_detection import detect_changes, preview_change, score_grid_cells
from src.detection.classifier import (
    extract_patches_from_gee, generate_pseudo_labels, train_classifier,
    classify_patches, classify_grid_cells,
)
from src.detection.risk_scorer import score_risk, save_risk_scores
from src.dashboard.map_builder import build_risk_map, save_risk_map
from src.dashboard.alert_report import generate_alert_report

config = load_config()
init_gee()

# Focused, live-validated analysis extent inside Old Oyo National Park.
analysis_bbox = [3.8, 8.5, 4.2, 8.9]
aoi = build_aoi(analysis_bbox)

2026-07-23 20:38:48.181 | SUCCESS  | src.ingestion.gee_download:init_gee:38 - GEE initialised — project: nigeria-forest-monitor
2026-07-23 20:38:48.183 | INFO     | src.ingestion.gee_download:build_aoi:50 - AOI: lon [3.8, 4.2] lat [8.5, 8.9]


## 2. Exploratory Data Analysis

Inspect Sentinel-1 availability, monthly composites, the project grid, and optional cached ACLED incidents.

In [2]:
eda_collection = get_s1_collection(aoi, "2024-01-01", "2024-03-01", config)
eda_composites = build_monthly_composites(
    eda_collection, "2024-01-01", "2024-03-01", aoi
)
print(f"Monthly composites: {len(eda_composites)}")

2026-07-23 20:38:49.182 | INFO     | src.ingestion.gee_download:get_s1_collection:86 - Found 11 Sentinel-1 images | 2024-01-01 → 2024-03-01
2026-07-23 20:38:50.273 | INFO     | src.ingestion.gee_download:build_monthly_composites:137 - Composite 2024-01 built from 7 images
2026-07-23 20:38:51.164 | INFO     | src.ingestion.gee_download:build_monthly_composites:137 - Composite 2024-02 built from 4 images
2026-07-23 20:38:51.164 | SUCCESS  | src.ingestion.gee_download:build_monthly_composites:140 - Built 2 monthly composites


Monthly composites: 2


In [3]:
if not eda_composites:
    raise RuntimeError("No EDA composites were available for the selected period")
preview_composite(eda_composites[0]["image"], aoi, "Old Oyo — January 2024")

Map(center=[8.700016581606397, 3.999999999999838], controls=(WidgetControl(options=['position', 'transparent_b…

In [4]:
grid = tag_zones(create_grid(config, zone="full"), config)
print(grid.groupby("zone")["cell_id"].count().rename("cells"))
preview_grid(grid, config, color_by_zone=True)

2026-07-23 20:38:54.452 | SUCCESS  | src.ingestion.grid:create_grid:61 - Grid created: 2304 cells | res=0.05° (~5.6 km) | zone=full
2026-07-23 20:38:54.585 | INFO     | src.ingestion.grid:tag_zones:89 - Zone tagging: {'outside': 1432, 'old_oyo_core': 336, 'kwara_border': 312, 'kainji_link': 224}


zone
kainji_link      224
kwara_border     312
old_oyo_core     336
outside         1432
Name: cells, dtype: int64


2026-07-23 20:38:55.824 | INFO     | src.ingestion.grid:preview_grid:233 - Grid map rendered: 2304 cells


Map(center=[9.0, 4.0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

In [5]:
try:
    acled_data = load_acled(config)
    incidents = filter_to_aoi(acled_data, config)
    incident_summary(incidents)
except FileNotFoundError as error:
    acled_data = pd.DataFrame()
    incidents = None
    print(f"Optional ACLED cache not found: {error}")
    print("Configure ACLED OAuth and fetch the cache separately if needed.")

Optional ACLED cache not found: ACLED data not found at C:\Users\USER\Desktop\nigeria-forest-monitor\data\acled\acled_nigeria.parquet. Run fetch_acled() first.
Configure ACLED OAuth and fetch the cache separately if needed.


In [6]:
raw_image = ee.Image(eda_collection.first()).clip(aoi)
preview_filter_comparison(raw_image, aoi, method="lee", band="VV")

2026-07-23 20:38:59.937 | INFO     | src.preprocessing.speckle_filter:preview_filter_comparison:218 - Comparison map ready — toggle layers to compare


Map(center=[8.700016581606397, 3.999999999999838], controls=(WidgetControl(options=['position', 'transparent_b…

In [7]:
baseline = build_baseline(aoi, config, filter_method="lee")
preview_baseline(baseline, aoi)

2026-07-23 20:39:00.216 | INFO     | src.preprocessing.baseline:build_baseline:51 - Building baseline: 2020-01-01 → 2022-12-31
2026-07-23 20:39:00.651 | INFO     | src.ingestion.gee_download:get_s1_collection:86 - Found 271 Sentinel-1 images | 2020-01-01 → 2022-12-31
2026-07-23 20:39:01.871 | INFO     | src.preprocessing.baseline:build_baseline:70 - Baseline collection: 271 images
2026-07-23 20:39:01.883 | SUCCESS  | src.preprocessing.baseline:build_baseline:77 - Baseline mosaic built from 271 images | filter: lee
2026-07-23 20:39:07.800 | INFO     | src.preprocessing.baseline:preview_baseline:231 - Baseline map ready — toggle VV / VH / RGB layers


Map(center=[8.700016581606397, 3.999999999999838], controls=(WidgetControl(options=['position', 'transparent_b…

## 3. Change Detection

Compare a monitoring-period composite with the historical baseline, clean small connected components, and inspect summary statistics.

In [8]:
monitor = get_s1_collection(aoi, "2025-01-01", "2025-02-01", config)
current = filter_composite(median_composite(monitor, aoi), method="lee")
detection = detect_changes(baseline, current, aoi, config)
detection["stats"]

2026-07-23 20:39:08.947 | INFO     | src.ingestion.gee_download:get_s1_collection:86 - Found 7 Sentinel-1 images | 2025-01-01 → 2025-02-01
2026-07-23 20:39:08.949 | INFO     | src.detection.change_detection:detect_changes:185 - Running change detection (band: VV)...
2026-07-23 20:39:17.448 | INFO     | src.detection.change_detection:threshold_change:94 - Adaptive threshold: 2.0σ × 0.6941 = ±1.3882 dB
2026-07-23 20:39:17.450 | INFO     | src.detection.change_detection:clean_change_mask:160 - Change mask cleaned (min patch: 10 pixels)
2026-07-23 20:39:27.789 | SUCCESS  | src.detection.change_detection:detect_changes:204 - Change detection complete | changed area: 50.69% of AOI


{'changed_px': 99075,
 'total_px': 195437,
 'changed_pct': 50.6942,
 'mean_ratio': -2.032942503567066,
 'max_ratio': 14.012316660673324}

In [9]:
preview_change(
    baseline,
    current,
    detection["change_mask"],
    detection["log_ratio"],
    aoi,
)

2026-07-23 20:39:33.941 | INFO     | src.detection.change_detection:preview_change:383 - Change map ready:
  Red pixels  = significant increase (structures/clearings)
  Blue pixels = significant decrease (vegetation loss)
  Toggle layers to compare baseline vs current


Map(center=[8.700016581606397, 3.999999999999838], controls=(WidgetControl(options=['position', 'transparent_b…

## 4. Feature Classifier

Extract a bounded six-feature table from changed locations and train the compact model.

The labels below are deterministic weak labels for software demonstration only. Operational training requires analyst-reviewed labels and independent validation.

In [10]:
features = extract_patches_from_gee(
    current,
    detection["change_mask"],
    aoi,
    config,
    n_patches=config["classifier"]["n_samples"],
)
if len(features) < 8:
    raise RuntimeError(
        f"Only {len(features)} feature rows were returned; at least 8 are required"
    )
labels = generate_pseudo_labels(features)
model = train_classifier(
    features,
    labels,
    config,
    save_path="models/sar_classifier_v1.pt",
)
print(f"Model trained on {len(features)} feature rows")

2026-07-23 20:39:34.188 | INFO     | src.detection.classifier:extract_patches_from_gee:128 - Sampling up to 200 changed locations at 100 m...
2026-07-23 20:40:18.685 | SUCCESS  | src.detection.classifier:extract_patches_from_gee:163 - Extracted 200 feature vectors at 100 m
2026-07-23 20:40:18.690 | INFO     | src.detection.classifier:generate_pseudo_labels:209 - Weak-label distribution: {'normal_forest': 98, 'clearing': 24, 'structure': 60, 'path_or_track': 18}
2026-07-23 20:40:21.991 | SUCCESS  | src.detection.classifier:train_classifier:298 - Training complete; best validation loss=1.0388
2026-07-23 20:40:21.996 | SUCCESS  | src.detection.classifier:save_model:387 - Model saved -> C:\Users\USER\Desktop\nigeria-forest-monitor\models\sar_classifier_v1.pt


Model trained on 200 feature rows


In [11]:
preview_count = min(10, len(features))
classification = classify_patches(model, features[:preview_count], config)
for index, (name, probability) in enumerate(
    zip(classification["class_names"], classification["probs"]), start=1
):
    print(f"Sample {index}: {name} ({probability.max() * 100:.1f}% confidence)")

Sample 1: normal_forest (72.0% confidence)
Sample 2: structure (42.8% confidence)
Sample 3: structure (42.9% confidence)
Sample 4: normal_forest (38.5% confidence)
Sample 5: normal_forest (71.4% confidence)
Sample 6: normal_forest (50.5% confidence)
Sample 7: normal_forest (57.8% confidence)
Sample 8: normal_forest (70.0% confidence)
Sample 9: normal_forest (50.6% confidence)
Sample 10: normal_forest (71.2% confidence)


## 5. Risk Scoring, Map, and Alert Report

Fuse grid-level change, classifier confidence, and optional ACLED proximity. Save a GeoPackage, interactive HTML map, and PDF alert report.

In [12]:
lon_min, lat_min, lon_max, lat_max = analysis_bbox
risk_grid = grid[
    grid["lon"].between(lon_min, lon_max)
    & grid["lat"].between(lat_min, lat_max)
].copy()
risk_grid = score_grid_cells(
    detection["change_mask"], detection["log_ratio"], risk_grid, config
)

classifier_scores = classify_grid_cells(model, current, risk_grid, config)
risk_grid = risk_grid.merge(classifier_scores, on="cell_id", how="left")
risk_grid["classifier_score"] = risk_grid["classifier_score"].fillna(0.0)

if incidents is not None and not incidents.empty:
    risk_grid = tag_incidents_to_grid(incidents, risk_grid, config)
    risk_grid = compute_proximity_scores(risk_grid, incidents, config)
else:
    risk_grid["acled_score"] = 0.0

scored_grid = score_risk(risk_grid, config)
scored_grid.sort_values("risk_score", ascending=False).head(10)

2026-07-23 20:40:22.748 | INFO     | src.detection.change_detection:score_grid_cells:317 - Scoring 64 grid cells for change...
2026-07-23 20:40:22.766 | INFO     | src.ingestion.grid:grid_to_ee_feature_collection:122 - Converted 64 cells to GEE FeatureCollection
2026-07-23 20:40:25.568 | SUCCESS  | src.detection.change_detection:score_grid_cells:351 - Grid scoring complete | 35 high-change cells
2026-07-23 20:40:25.574 | INFO     | src.ingestion.grid:grid_to_ee_feature_collection:122 - Converted 64 cells to GEE FeatureCollection
2026-07-23 20:40:26.908 | SUCCESS  | src.detection.risk_scorer:score_risk:59 - Risk scoring complete: 0 alert cells


,cell_id,lat,lon,lat_min,lon_min,lat_max,lon_max,zone,geometry,changed_frac,mean_intensity,change_score,classifier_score,acled_score,risk_score,risk_level,alert,risk_rank
28,840,8.675,4.025,8.65,4.00,8.70,4.05,old_oyo_core,"POLYGON ((4.05000 8.65000, 4.05000 8.70000, 4....",0.977359,2.371152,0.969804,0.471117,0.0,0.552813,moderate,False,1
20,792,8.625,4.025,8.60,4.00,8.65,4.05,old_oyo_core,"POLYGON ((4.05000 8.60000, 4.05000 8.65000, 4....",0.975380,2.449942,1.000000,0.433502,0.0,0.551726,moderate,False,2
56,1028,8.875,3.825,8.85,3.80,8.90,3.85,old_oyo_core,"POLYGON ((3.85000 8.85000, 3.85000 8.90000, 3....",0.988813,2.352118,0.973294,0.374754,0.0,0.520481,moderate,False,3
12,744,8.575,4.025,8.55,4.00,8.60,4.05,old_oyo_core,"POLYGON ((4.05000 8.55000, 4.05000 8.60000, 4....",0.990141,2.306394,0.955654,0.333097,0.0,0.498846,moderate,False,4
29,841,8.675,4.075,8.65,4.05,8.70,4.10,old_oyo_core,"POLYGON ((4.10000 8.65000, 4.10000 8.70000, 4....",0.963753,2.187468,0.882222,0.407687,0.0,0.495579,moderate,False,5
40,932,8.775,3.825,8.75,3.80,8.80,3.85,old_oyo_core,"POLYGON ((3.85000 8.75000, 3.85000 8.80000, 3....",0.977479,2.121770,0.867913,0.379697,0.0,0.480059,moderate,False,6
4,696,8.525,4.025,8.50,4.00,8.55,4.05,old_oyo_core,"POLYGON ((4.05000 8.50000, 4.05000 8.55000, 4....",0.961113,2.157430,0.867724,0.348165,0.0,0.468947,moderate,False,7
63,1035,8.875,4.175,8.85,4.15,8.90,4.20,old_oyo_core,"POLYGON ((4.20000 8.85000, 4.20000 8.90000, 4....",0.943734,1.992955,0.787077,0.429494,0.0,0.465154,moderate,False,8
13,745,8.575,4.075,8.55,4.05,8.60,4.10,old_oyo_core,"POLYGON ((4.10000 8.55000, 4.10000 8.60000, 4....",0.985250,2.096774,0.864507,0.317038,0.0,0.456766,moderate,False,9
23,795,8.625,4.175,8.60,4.15,8.65,4.20,old_oyo_core,"POLYGON ((4.20000 8.60000, 4.20000 8.65000, 4....",0.940823,2.000146,0.787481,0.394508,0.0,0.453070,moderate,False,10


In [13]:
scores_path = save_risk_scores(scored_grid, config)
risk_map = build_risk_map(scored_grid, config, incidents)
map_path = save_risk_map(risk_map, config)
report_path = generate_alert_report(scored_grid, config)

print(f"Scores: {scores_path}")
print(f"Map: {map_path}")
print(f"Report: {report_path}")
risk_map

2026-07-23 20:40:28.884 | SUCCESS  | src.detection.risk_scorer:save_risk_scores:87 - Risk scores saved -> C:\Users\USER\Desktop\nigeria-forest-monitor\data\processed\risk_scores.gpkg
2026-07-23 20:40:28.937 | SUCCESS  | src.dashboard.map_builder:save_risk_map:78 - Risk map saved -> C:\Users\USER\Desktop\nigeria-forest-monitor\reports\risk_map.html
2026-07-23 20:40:28.963 | SUCCESS  | src.dashboard.alert_report:generate_alert_report:93 - Alert report saved -> C:\Users\USER\Desktop\nigeria-forest-monitor\reports\forest_alert_20260723_194028.pdf


Scores: C:\Users\USER\Desktop\nigeria-forest-monitor\data\processed\risk_scores.gpkg
Map: C:\Users\USER\Desktop\nigeria-forest-monitor\reports\risk_map.html
Report: C:\Users\USER\Desktop\nigeria-forest-monitor\reports\forest_alert_20260723_194028.pdf
